<a href="https://colab.research.google.com/github/aot-abhishekyadav/Machine-Learning-Fundamentals/blob/main/19-IterativeImputer/iteraive-imputer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Iterative Impuation kaa laogritham yai hai ki vo phele jin rows mai missing values hoti hai usn rows kii means values nikal kai uss miss values ko fill kar deta hai aur phir next step mai kisi ek row ko leta hai aur usme jo miss vallue thi usko vapas nan kar deta hai phir next step mai jo row kii value ko nan karta hai vo humari target column ban jaati hai aur baki remaning column humare input column ban jaate hai aur phir linear model laga kai uss value ko predict karte hai

## When we have to use Iterative Imputation
### 1. MCAR (Missing Completely At Random)
iska mtlb hai ki data mai phele sai value missing hai

### 2. MAR(Missing At Random)
iska mtlb hai ki data mai apne kuch information jaan bhuj kai fill nhi kari jaise optional dat aur kiski ko apna kuch personal information nhi share karna ho

### 3. MNAR(Missing Not At Random)
iska mtlb hai kii data ko jaan bhuj kai remove kiya gya hai

### Adv
1. Accurate

2. Other features se missing values predict karta hai.

3. Feature relationships ko preserve kar sakta hai.

4. Multiple missing columns wale datasets me useful.

5. Simple mean/median se potentially better estimates de sakta hai.

### Disadv
1. slow

2. Take more memory

3. Computationally expensive.

4. Large datasets par slow ho sakta hai.

5. Overfitting ka risk.

6. Results estimator aur dataset par depend karte hain.

7. Proper train/test handling na ho to data leakage ho sakta hai.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/50_Startups.csv')

In [ ]:
data.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [ ]:
df = np.round(data[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)

In [ ]:
df.head()

,R&D Spend,Administration,Marketing Spend,Profit
0,17.0,14.0,47.0,19.0
1,16.0,15.0,44.0,19.0
2,15.0,10.0,41.0,19.0
3,14.0,12.0,38.0,18.0
4,14.0,9.0,37.0,17.0


In [ ]:
np.random.seed(9)
df = df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [ ]:
df = df.iloc[:,0:-1].copy()
df # Exceeding profit column

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


creating Nan value by itself

In [ ]:
df.iloc[1,0] = np.nan
df.iloc[3,1] = np.nan
df.iloc[-1,-1] = np.nan

In [ ]:
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


In [ ]:
# Step 1 - Impute all missing values with mean of respective col

df0 = pd.DataFrame()

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

In [ ]:
# 0th Iteration
df0

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


In [ ]:
# Remove the col1 imputed value
df1 = df0.copy()

df1.iloc[1,0] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


In [ ]:
# Use first 3 rows to build a model and use the last for prediction

X = df1.iloc[[0,2,3,4],1:3]
X

,Administration,Marketing Spend
21,15.00,30.00
2,10.00,41.00
14,11.25,26.00
44,15.00,29.25


In [ ]:
y = df1.iloc[[0,2,3,4],0]
y

,R&D Spend
21,8.0
2,15.0
14,12.0
44,2.0


In [ ]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[1,1:].values.reshape(1,2))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.14158651])

In [ ]:
df1.iloc[1,0] = 23.14

In [ ]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


In [ ]:
# Remove the col2 imputed value

df1.iloc[3,1] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.0,30.00
37,23.14,5.0,20.00
2,15.00,10.0,41.00
14,12.00,NaN,26.00
44,2.00,15.0,29.25


In [ ]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[[0,1,2,4],[0,2]]
X

,R&D Spend,Marketing Spend
21,8.00,30.00
37,23.14,20.00
2,15.00,41.00
44,2.00,29.25


In [ ]:
y = df1.iloc[[0,1,2,4],1]
y

,Administration
21,15.0
37,5.0
2,10.0
44,15.0


In [ ]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[3,[0,2]].values.reshape(1,2))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.06331285])

In [ ]:
df1.iloc[3,1] = 11.06
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,29.25


In [ ]:
# Remove the col3 imputed value
df1.iloc[4,-1] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.0
37,23.14,5.00,20.0
2,15.00,10.00,41.0
14,12.00,11.06,26.0
44,2.00,15.00,NaN


In [ ]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[0:4,0:2]
X

,R&D Spend,Administration
21,8.00,15.00
37,23.14,5.00
2,15.00,10.00
14,12.00,11.06


In [ ]:
y = df1.iloc[0:4,-1]
y

,Marketing Spend
21,30.0
37,20.0
2,41.0
14,26.0


In [ ]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[4,0:2].values.reshape(1,2))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([31.56351448])

In [ ]:
df1.iloc[4,-1] = 31.56
# After 1st Iteration
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


In [ ]:
# Subtract 0th iteration from 1st iteration

df1 - df0

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,13.89,0.00,0.00
2,0.00,0.00,0.00
14,0.00,-0.19,0.00
44,0.00,0.00,2.31


In [ ]:
df2 = df1.copy()

df2.iloc[1,0] = np.nan

df2

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.06,26.00
44,2.0,15.00,31.56


In [ ]:
X = df2.iloc[[0,2,3,4],1:3]
y = df2.iloc[[0,2,3,4],0]

In [ ]:
X

,Administration,Marketing Spend
21,15.00,30.00
2,10.00,41.00
14,11.06,26.00
44,15.00,31.56


In [ ]:
y

,R&D Spend
21,8.0
2,15.0
14,12.0
44,2.0


In [ ]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[1,1:].values.reshape(1,2))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.78627207])

In [ ]:
df2.iloc[1,0] = 23.78

In [ ]:
df2.iloc[3,1] = np.nan
df2

,R&D Spend,Administration,Marketing Spend
21,8.00,15.0,30.00
37,23.78,5.0,20.00
2,15.00,10.0,41.00
14,12.00,NaN,26.00
44,2.00,15.0,31.56


In [ ]:
X = df2.iloc[[0,1,2,4],[0,2]]
y = df2.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[3,[0,2]].values.reshape(1,2))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.22020174])

In [ ]:
df2.iloc[3,1] = 11.22

In [ ]:
df2.iloc[4,-1] = np.nan
df2

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.0
37,23.78,5.00,20.0
2,15.00,10.00,41.0
14,12.00,11.22,26.0
44,2.00,15.00,NaN


In [ ]:
X = df2.iloc[0:4,0:2]
X

,R&D Spend,Administration
21,8.00,15.00
37,23.78,5.00
2,15.00,10.00
14,12.00,11.22


In [ ]:
y = df2.iloc[0:4,-1]
y

,Marketing Spend
21,30.0
37,20.0
2,41.0
14,26.0


In [ ]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[4,0:2].values.reshape(1,2))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([38.87979054])

In [ ]:
df2.iloc[4,-1] = 38.87

In [ ]:
df2

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.78,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.22,26.00
44,2.00,15.00,38.87


In [ ]:
df2 - df1

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,0.64,0.00,0.00
2,0.00,0.00,0.00
14,0.00,0.16,0.00
44,0.00,0.00,7.31


In [ ]:
df3 = df2.copy()

df3.iloc[1,0] = np.nan

df3

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.22,26.00
44,2.0,15.00,38.87


In [ ]:
X = df3.iloc[[0,2,3,4],1:3]
y = df3.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[1,1:].values.reshape(1,2))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([26.82082209])

In [ ]:
df3.iloc[1,0] = 26.82

In [ ]:
df3.iloc[3,1] = np.nan
df3

,R&D Spend,Administration,Marketing Spend
21,8.00,15.0,30.00
37,26.82,5.0,20.00
2,15.00,10.0,41.00
14,12.00,NaN,26.00
44,2.00,15.0,38.87


In [ ]:
X = df3.iloc[[0,1,2,4],[0,2]]
y = df3.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[3,[0,2]].values.reshape(1,2))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([12.22106654])

In [ ]:
df3.iloc[3,1] = 12.22

In [ ]:
df3.iloc[4,-1] = np.nan

X = df3.iloc[0:4,0:2]
y = df3.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[4,0:2].values.reshape(1,2))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([62.65086843])

In [ ]:
df3.iloc[4,-1] = 62.65

In [ ]:
df3

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,26.82,5.00,20.00
2,15.00,10.00,41.00
14,12.00,12.22,26.00
44,2.00,15.00,62.65


In [ ]:
df3 - df2


,R&D Spend,Administration,Marketing Spend
21,0.00,0.0,0.00
37,3.04,0.0,0.00
2,0.00,0.0,0.00
14,0.00,1.0,0.00
44,0.00,0.0,23.78
